# 综合实训 · OpenMP × NEON 协同优化矩阵乘法 GEMM

**所属**：《并行计算》第五章 · OpenMP 多线程编程　|　**难度**：⭐⭐⭐⭐⭐ 综合　|　**预计时长**：60–90 分钟

> **实验说明**
> 1. 本实验是第五章的**综合实训**，也是第三章 GEMM 实训的延续：并行化的对象不是朴素串行实现，而是第三章已经完成 NEON 优化的单线程实现，即在数据级并行（DLP）之上叠加线程级并行（TLP）。
> 2. 五个版本为递增关系：**v1 两条基准 → v2 朴素 OpenMP → v3 OpenMP × Cache 分块 NEON → v4 同一指令用于打包版本（存在数据竞争）→ v5 线程私有缓冲区修复**。每一版在前一版基础上新增一个实现，输出表格相应增加一行。
> 3. 除版本递进外，本实验还包含第二条递进路径：第 11 节保持代码不变、仅改变线程数，绘制**强扩展性曲线**，这是 OpenMP 章特有的研究方法。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 本实验依赖 **ARM(aarch64/arm64) + NEON + OpenMP**，须在华为鲲鹏处理器上运行，编译时需添加 `-fopenmp`。
> 6. ⚠️ **第 8 节的 v4 将按设计报告 `FAIL`**，这不是环境故障，而是本实验的核心教学素材，请勿跳过。
> 7. 建议先完成**第三章综合实训 GEMM**。


## 🎯 学习目标

完成本任务的学习后，学生应能够：

- 区分**指令级并行（ILP）**、**数据级并行（DLP/SIMD）**与**线程级并行（TLP）**，并说明三者为何呈相乘关系
- 掌握 `#pragma omp parallel for` 的用法，以及 `num_threads`、`schedule`、`collapse`、`default(none)`、`shared`、`private` 各子句的语义
- 辨识**加速比的局限性**：说明加速比为何只能反映"相对自身串行版本的改善"，以及比较不同实现时为何必须采用绝对性能指标
- 掌握判断"循环能否直接并行"的三条判据，特别是最易被忽略的第三条——迭代内的临时工作区必须为线程私有
- 通过实验观察**数据竞争**的典型特征：同一核函数单线程通过校验、添加并行指令后校验失败，且不产生崩溃与编译警告
- 理解 `collapse(2)` 如何通过扩大迭代空间改善**负载均衡**，以及 `schedule` 各策略的适用场景
- 独立完成**强扩展性研究**：正确选取基准、绘制加速比与并行效率曲线，并从任务粒度、NUMA 拓扑与内存带宽三个方向解释效率下降
- 建立"先验证正确性、再讨论性能"的并行程序开发规范


## 🗺️ 学习路径

1. **准备阶段**：明确第三章单核优化的两个结果特征——浮点算力已接近单核可达上限，而内存带宽占用率很低。前者说明单核优化空间有限，后者说明存储系统仍有充裕余量可供多线程使用
2. **v1 · 两条基准**：`gemm_serial_no_vec`（标量基准）与 `gemm_neon_1t`（第三章最优单线程实现）
   → 明确并行化的对象与两种加速比的含义
3. **v2 · 朴素 OpenMP**：对标量三重循环添加 `#pragma omp parallel for`
   → 考察"加速比数值可观但绝对性能低下"这一现象及其成因
4. **v3 · OpenMP × Cache 分块 NEON**：对第三章的 Cache 分块版本（**不含临时缓冲区**）添加同一指令
   → 用三条判据逐条验证；掌握 `collapse(2)` 与 `schedule`
5. **v4 · 同一指令用于打包版本**：打包版本内部持有 `packed_B` 暂存区
   → **数据竞争**：校验失败，且无崩溃、无编译警告。与 v3 的唯一差异是核函数是否持有内部状态
6. **v5 · 线程私有缓冲区**：将缓冲区的分配移入并行域
   → 掌握 `omp parallel { omp for }` 的拆分写法
7. **可视化与扩展性研究**：先绘制六个实现的对比图，再固定代码、仅改变线程数，绘制强扩展性与并行效率曲线
8. **🚀 扩展实验**：将 OpenMP 应用于第三章的 8×8 双打包微内核，并探究 NUMA first-touch 与线程绑定的影响


## 1. 背景与动机：本实验的起点

第三章的综合实训在**单核**范围内对 GEMM 做了逐级优化，最终实现具有两个特征：

<!--
| 指标 | 第三章最终实现的状态 | 对本章的意义 |
|---|---|---|
| 浮点算力 | 已接近单核可达上限，进一步提升需要更大的微内核，收益有限 | 单核优化空间基本用尽 |
| 内存带宽 | 因 Cache 分块与内存打包，访存量大幅下降，带宽占用率很低 | **存储系统仍有充裕余量** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">指标</th>
      <th style="text-align: left;">第三章最终实现的状态</th>
      <th style="text-align: left;">对本章的意义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">浮点算力</td>
      <td style="text-align: left;">已接近单核可达上限，进一步提升需要更大的微内核，收益有限</td>
      <td style="text-align: left;">单核优化空间基本用尽</td>
    </tr>
    <tr>
      <td style="text-align: left;">内存带宽</td>
      <td style="text-align: left;">因 Cache 分块与内存打包，访存量大幅下降，带宽占用率很低</td>
      <td style="text-align: left;"><strong>存储系统仍有充裕余量</strong></td>
    </tr>
  </tbody>
</table>

这两个特征共同决定了本章的可行性：**一个算力接近饱和、而带宽占用很低的核函数，是线程级并行的理想对象**。若将其复制到 $T$ 个核心上，带宽需求也只是随 $T$ 线性增长，在核心数不太大时不会立即成为瓶颈。

> 上述判断是定性的。**具体数值请以本实验 v1 与第 11 节的实测结果为准**。

本章要回答的问题是：**理论上的 $T$ 倍加速，实际能够获得多少？未能获得的部分损失在何处？**


## 2. 三个层级的并行

现代处理器的性能来自三个相互正交的并行层级：

<!--
| 层级 | 全称 | 硬件载体 | 使用方式 | 本课程对应内容 |
|---|---|---|---|---|
| 指令级 | ILP (Instruction-Level Parallelism) | 超标量发射、多条流水线 | 多累加器打破依赖链，提高 `fmla` 指令占比 | 第三章 v3 |
| 数据级 | DLP / SIMD | 128 位向量寄存器（NEON） | 一条指令处理 4 个 float | 第三章 v2–v5 |
| 线程级 | TLP (Thread-Level Parallelism) | 多个物理核心 | OpenMP 将迭代分配给多个线程 | **本章** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">层级</th>
      <th style="text-align: left;">全称</th>
      <th style="text-align: left;">硬件载体</th>
      <th style="text-align: left;">使用方式</th>
      <th style="text-align: left;">本课程对应内容</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">指令级</td>
      <td style="text-align: left;">ILP (Instruction-Level Parallelism)</td>
      <td style="text-align: left;">超标量发射、多条流水线</td>
      <td style="text-align: left;">多累加器打破依赖链，提高 <code>fmla</code> 指令占比</td>
      <td style="text-align: left;">第三章 v3</td>
    </tr>
    <tr>
      <td style="text-align: left;">数据级</td>
      <td style="text-align: left;">DLP / SIMD</td>
      <td style="text-align: left;">128 位向量寄存器（NEON）</td>
      <td style="text-align: left;">一条指令处理 4 个 float</td>
      <td style="text-align: left;">第三章 v2–v5</td>
    </tr>
    <tr>
      <td style="text-align: left;">线程级</td>
      <td style="text-align: left;">TLP (Thread-Level Parallelism)</td>
      <td style="text-align: left;">多个物理核心</td>
      <td style="text-align: left;">OpenMP 将迭代分配给多个线程</td>
      <td style="text-align: left;"><strong>本章</strong></td>
    </tr>
  </tbody>
</table>

三者对性能的贡献是**相乘**关系：

$$\text{理论峰值} = \underbrace{(\text{每周期 FMA 条数})}_{\text{ILP}} \times \underbrace{(\text{每条 FMA 的通道数})}_{\text{DLP}} \times \underbrace{(\text{核心数})}_{\text{TLP}} \times 2 \times f$$

> **⭐ 由相乘关系可以直接推出两条结论：**
>
> 1. **在哪一个层级上并行化，决定了性能能够被放大多少倍。** 若单核实现只达到峰值的 1%，即便扩展到 64 个核心，整体仍不足峰值的 64%；
> 2. **并行化不能弥补串行实现的低效**，它只会将同样的低效复制 $T$ 份。因此优化的正确顺序是**先单核、后多核**。
>
> 第 6 节将用本次运行的数据对这两条结论予以验证。


## 3. 核心 OpenMP 指令与概念

### 3.1 指令的完整形式

```c
#pragma omp parallel for num_threads(T) collapse(2) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
for (int ii = 0; ii < M; ii += BLOCK_M) {
  for (int jj = 0; jj < N; jj += BLOCK_N) {
    ...                       // 循环变量 ii、jj 自动为 private
  }
}
```

- **`parallel`** 创建线程团队；**`for`** 将紧随其后的循环迭代分配给团队成员。二者常合写为 `parallel for`；但当需要在并行域内执行**每线程一次**的准备工作（例如分配私有缓冲区）时，必须拆分为 `omp parallel { ... omp for ... }`。
- **`num_threads(T)`** 指定本次使用的线程数，亦可通过环境变量 `OMP_NUM_THREADS` 或函数 `omp_set_num_threads()` 设置。
- **`collapse(n)`** 将 $n$ 层**完美嵌套**的循环折叠为一个迭代空间，用于**扩大并行度**——仅并行外层循环时任务数可能过少，无法在多线程间均衡分配。
- **`schedule(kind[, chunk])`** 指定迭代的分配方式：`static` 同步开销最低，适用于各迭代工作量相等的场合；`dynamic` 按需领取，适用于工作量不均的场合；`guided` 为块大小递减的动态调度。
- **`default(none)`** 强制为每个变量显式声明共享属性，否则编译报错。本实验全程使用该子句，以便在编译期发现遗漏的私有声明。

### 3.2 数据共享属性

<!--
| 属性 | 含义 | 默认适用范围 |
|---|---|---|
| `shared` | 所有线程访问**同一份**内存 | 并行域**外**声明的变量（默认行为，也是常见错误的来源） |
| `private` | 每个线程持有一份**未初始化**的副本 | 并行循环的循环变量 |
| `firstprivate` | 每线程一份副本，以进入并行域前的值初始化 | — |
| `reduction(op:var)` | 每线程一份局部副本，退出时按 `op` 合并 | 求和、求极值等规约运算 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">属性</th>
      <th style="text-align: left;">含义</th>
      <th style="text-align: left;">默认适用范围</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>shared</code></td>
      <td style="text-align: left;">所有线程访问<strong>同一份</strong>内存</td>
      <td style="text-align: left;">并行域<strong>外</strong>声明的变量（默认行为，也是常见错误的来源）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>private</code></td>
      <td style="text-align: left;">每个线程持有一份<strong>未初始化</strong>的副本</td>
      <td style="text-align: left;">并行循环的循环变量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>firstprivate</code></td>
      <td style="text-align: left;">每线程一份副本，以进入并行域前的值初始化</td>
      <td style="text-align: left;">—</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>reduction(op:var)</code></td>
      <td style="text-align: left;">每线程一份局部副本，退出时按 <code>op</code> 合并</td>
      <td style="text-align: left;">求和、求极值等规约运算</td>
    </tr>
  </tbody>
</table>

> **⚠️ 一类典型错误**：在并行域**外部**分配一块临时工作区，而在并行域**内部**写入该工作区。指针变量本身属于 `shared`，因此所有线程写入同一块内存。此类错误不产生编译警告，程序也不会崩溃，但计算结果错误。

### 3.3 判断循环可并行性的三条判据

对循环体的每一次迭代逐条检查：

1. **不存在跨迭代的数据依赖**：迭代 $i$ 不读取迭代 $i-1$ 写出的结果；否则需要修改算法或使用 `reduction`；
2. **写入区域互不重叠**：任意两次迭代写入的内存地址集合不相交；否则需要 `atomic` 或 `critical`，而这通常意味着应当更换并行维度；
3. **迭代内的临时工作区为线程私有**：所有暂存缓冲区、状态变量、静态变量均须每线程一份。

> 就 GEMM 而言，`i` 与 `j` 方向天然满足前两条（每个 $C_{ij}$ 仅被一次迭代写入），而 `k` 方向不满足第一条（它是规约方向，所有 $k$ 累加到同一个 $C_{ij}$）。因此本实验**始终只并行 `i`/`j` 方向，不并行 `k` 方向**。
>
> 第三条判据在教材中往往论述较少，却是工程实践中错误率最高的一条。本实验以 v3 与 v4 的对照专门考察这一条。


## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")

# OpenMP 可用性与核心数检测
NCORE = os.cpu_count() or 1
HAS_OMP = False
if CC:
    _probe = "#include <omp.h>\n#include <stdio.h>\nint main(){printf(\"%d\", omp_get_max_threads());return 0;}"
    open("/tmp/_omp_probe.c", "w").write(_probe)
    _r = subprocess.run(f"{CC} -fopenmp /tmp/_omp_probe.c -o /tmp/_omp_probe",
                        shell=True, capture_output=True, text=True)
    if _r.returncode == 0:
        HAS_OMP = True
        NCORE = int(subprocess.run(["/tmp/_omp_probe"], capture_output=True,
                                   text=True).stdout.strip() or NCORE)

NT = NCORE                      # 本实验默认使用的线程数
print("逻辑核心:", os.cpu_count(), " | OpenMP 可见核心:", NCORE)
print("OpenMP  :", "可用" if HAS_OMP else "不可用")

if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif not HAS_OMP:
    print("\n⚠️  编译器不支持 -fopenmp，请安装支持 OpenMP 的 GCC。")
else:
    print(f"\n✅ 环境就绪：ARM + NEON + OpenMP，可用 {NT} 个线程，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。
    与第三章相比唯一的变化：加上 -fopenmp。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -fopenmp -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC -fopenmp"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "threads", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows


def parse_gflops(text):
    """取出第 4 列 GFLOPS。表格列序为 方法|耗时|加速比|GFLOPS|校验。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 4:
            continue
        name = cells[0]
        if name.lower() in ("method", "threads", "方法") or set(name) <= set("-: "):
            continue
        mg = re.search(r"[-+]?\d*\.?\d+", cells[3])
        if not mg:
            continue
        rows.append({"method": name, "gflops": float(mg.group())})
    return rows


def parse_check(text):
    """取出校验列，用于在图上标出 FAIL 的版本。"""
    st = {}
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 5:
            continue
        st[cells[0]] = cells[4]
    return st

In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title="", checks=None):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    # 校验未通过的版本一律涂成橙色并加剖面线，避免被误读成"最优"
    hatch = [None] * len(sp)
    if checks:
        for i, n in enumerate(names):
            if checks.get(n, "").upper() == "FAIL":
                colors[i] = "#E8A33D"
                hatch[i] = "//"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    for b, h in zip(bars, hatch):
        if h:
            b.set_hatch(h)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


def plot_gflops(rows, title="", checks=None):
    if not rows:
        print("未解析到可绘制的 GFLOPS。")
        return
    names = [r["method"] for r in rows]
    gf = [r["gflops"] for r in rows]
    colors = ["#295E96"] * len(gf)
    valid = [i for i, n in enumerate(names)
             if not (checks and checks.get(n, "").upper() == "FAIL")]
    if valid:
        colors[max(valid, key=lambda i: gf[i])] = "#C7000B"
    if checks:
        for i, n in enumerate(names):
            if checks.get(n, "").upper() == "FAIL":
                colors[i] = "#E8A33D"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, gf, color=colors)
    if checks:
        for b, n in zip(bars, names):
            if checks.get(n, "").upper() == "FAIL":
                b.set_hatch("//")
    for b, g in zip(bars, gf):
        plt.text(
            b.get_x() + b.get_width() / 2,
            g,
            f"{g:.1f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("GFLOPS")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


def plot_scaling(rows, title=""):
    """强扩展性：左轴加速比（含理想线），右轴并行效率。"""
    ts = [int(r["method"]) for r in rows]
    sp = [r["speedup"] for r in rows]
    eff = [s / t * 100 for s, t in zip(sp, ts)]
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax1.plot(ts, ts, ls="--", c="gray", lw=1, label="Ideal (linear)")
    ax1.plot(ts, sp, "o-", c="#C7000B", lw=2, label="Measured speedup")
    ax1.set_xlabel("Threads")
    ax1.set_ylabel("Speedup (x)")
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(ts)
    ax1.set_xticklabels([str(t) for t in ts])
    ax2 = ax1.twinx()
    ax2.plot(ts, eff, "s--", c="#295E96", lw=1.5, label="Efficiency")
    ax2.set_ylabel("Parallel efficiency (%)")
    ax2.set_ylim(0, 110)
    ax2.axhline(100, ls=":", c="#295E96", lw=0.8)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
    plt.title(title)
    plt.tight_layout()
    plt.show()
    print(f'{"Threads":>8} {"Speedup":>9} {"Efficiency":>11}')
    for t, s, e in zip(ts, sp, eff):
        print(f"{t:>8} {s:>8.2f}x {e:>10.1f}%")

In [ ]:
# 创建源代码目录
!mkdir -p src_gemm

## 5. v1 · 两条基准

在实施并行化之前，必须先确定**基准**。v1 包含两个函数，均取自第三章：

<!--
| 函数 | 来源 | 说明 |
|---|---|---|
| `gemm_serial_no_vec` | 第三章 v1 | 关闭自动向量化的标量三重循环，作为 **1.00×** 的性能基准 |
| `gemm_neon_1t` | 第三章 v5 | 4×4 寄存器分块 + B 块打包的**单线程**实现，本章并行化的对象 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">来源</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>gemm_serial_no_vec</code></td>
      <td style="text-align: left;">第三章 v1</td>
      <td style="text-align: left;">关闭自动向量化的标量三重循环，作为 <strong>1.00×</strong> 的性能基准</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>gemm_neon_1t</code></td>
      <td style="text-align: left;">第三章 v5</td>
      <td style="text-align: left;">4×4 寄存器分块 + B 块打包的<strong>单线程</strong>实现，本章并行化的对象</td>
    </tr>
  </tbody>
</table>

### 💡 关注点：两种加速比的含义不同

后续所有多线程版本都将同时与这两条基准比较：

- 相对 `Serial (No-Vec)` 的加速比，反映的是 **SIMD 与多线程的合成效果**；
- 相对 `NEON 1-Thread` 的加速比，才是**线程级并行本身的贡献**。

请在运行后记录表中 `NEON 1-Thread` 一行的 GFLOPS 值，后续各节的分析均以该值为参照。

### 代码要点（沿用第三章的约定）
- 每次计时前执行 `memset(C_test, 0, bytes_C)`，且该语句位于计时区间之外；
- `check_result` 采用随累加长度 $K$ 缩放的相对容差 $2\times10^{-8}K\max|C_{ref}|$；
- 优化版本重复 `NTIMES = 5` 次取平均，串行基准单独设 `NTIMES_BASE = 1`；
- 命令行第 4 个参数为线程数，**取 0 或省略表示使用全部可见核心**；程序同时打印 `omp_get_max_threads()`，便于与 `lscpu` 的结果核对。

> 📌 请注意 `gemm_neon_1t` 中的 `aligned_alloc` 语句：整个函数只申请了**一块** `packed_B` 缓冲区。在单线程条件下这一写法并无问题，第 8 节将对此展开分析。

In [ ]:
%%writefile src_gemm/omp_gemm_v1.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP v1: Two Starting Lines -- Serial vs Single-Thread NEON (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_v1.c", "src_gemm/omp_gemm_v1")
out_v1 = run_bin(BIN, 1024, 1024, 1024, NT)

## 6. v2 · 朴素 OpenMP

在 v1 的基础上**新增 `gemm_omp_naive` 函数**：对**标量**三重循环添加一条并行指令。

```c
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
for (int i = 0; i < M; i++) {          // 每个 i 对应 C 的一整行，彼此独立
  for (int j = 0; j < N; j++) {
    float sum = 0.0f;
    for (int k = 0; k < K; k++) sum += A[i*K+k] * B[k*N+j];
    C[i*N+j] = sum;
  }
}
```

按第 3 节的三条判据检查：迭代 $i$ 只写 `C` 的第 $i$ 行（判据 ② 成立）；`sum` 是循环体内的局部变量，自动为私有（判据 ③ 成立）；行与行之间不存在依赖（判据 ① 成立）。**因此该并行化是正确的**，且无需对原有代码作任何修改。

### ⚠️ 关注点：请分别按"加速比"与"GFLOPS"两列对各行排序

运行后请完成三项观察：

1. `OMP Naive` 相对串行基准取得了**数倍到十余倍**的加速比，从"并行化是否生效"的角度看是成功的；
2. 但其 **GFLOPS 明显低于单线程的 `NEON 1-Thread`**——即一个占用全部核心的并行程序，性能低于一个单线程程序；
3. 计算它的**并行效率**（加速比 ÷ 线程数）。该值通常**远低于** 100%，也明显低于后续 v3、v5 的并行效率。

### 💡 现象解释：低效实现的两重代价

第 3 点是最容易被忽略的。朴素三重循环中 `B[k*N+j]` 的访问步长为 $N$，缓存行利用率极低，属于**访存受限**的实现。访存受限意味着：

- **绝对性能低**：单个线程的算力就已被访存拖住；
- **扩展性也差**：多个线程同时以低效方式访问内存，很快就会耗尽存储系统的供给能力，因此加速比达不到线程数。

> **🎓 结论**：加速比是**相对量**，其分母由使用者自行选定，因此它只能回答"相对自身的串行版本改善了多少"，**不能回答"这是否是一个好的实现"**。比较不同实现时，唯一可靠的指标是绝对性能（GFLOPS 或耗时）。
>
> 加速比真正可信的用法只有一种：**固定同一份代码，仅改变线程数**——这正是第 11 节强扩展性研究所采用的方法。

In [ ]:
%%writefile src_gemm/omp_gemm_v2.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP v2: + Naive OpenMP on the Scalar Triple Loop (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive OpenMP on the scalar triple loop
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_naive(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_omp = (end - start) / NTIMES;
  report("OMP Naive", t_omp, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_v2.c", "src_gemm/omp_gemm_v2")
out_v2 = run_bin(BIN, 1024, 1024, 1024, NT)

## 7. v3 · OpenMP × Cache 分块 NEON

既然并行化标量实现不可行，转而并行化 NEON 实现。**先从第三章不含打包步骤的 Cache 分块版本入手**（`microkernel_4x4` + 三方向分块）。

```c
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
for (int ii = 0; ii < M; ii += BLOCK_M) {
  for (int jj = 0; jj < N; jj += BLOCK_N) {
    ...                                   // 清零本块，随后沿 K 累加
    microkernel_4x4(...);
  }
}
```

### 💡 关注点：为何本版无需任何额外修改

逐条对照第 3 节的三条判据：

<!--
| 判据 | 本例的情况 | 结论 |
|---|---|---|
| ① 不存在跨迭代依赖 | 每个 `(ii, jj)` 任务独立计算自己的 C 块，不读取其他任务的结果 | 满足 |
| ② 写入区域不重叠 | 任务 `(ii, jj)` 仅写入 `C[ii..ii+63][jj..jj+63]`，块与块严格不相交 | 满足 |
| ③ 临时工作区线程私有 | **本版不含任何临时工作区**——`microkernel_4x4` 在寄存器中完成计算，A、B 均为就地读取 | **自动满足** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">判据</th>
      <th style="text-align: left;">本例的情况</th>
      <th style="text-align: left;">结论</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">① 不存在跨迭代依赖</td>
      <td style="text-align: left;">每个 <code>(ii, jj)</code> 任务独立计算自己的 C 块，不读取其他任务的结果</td>
      <td style="text-align: left;">满足</td>
    </tr>
    <tr>
      <td style="text-align: left;">② 写入区域不重叠</td>
      <td style="text-align: left;">任务 <code>(ii, jj)</code> 仅写入 <code>C[ii..ii+63][jj..jj+63]</code>，块与块严格不相交</td>
      <td style="text-align: left;">满足</td>
    </tr>
    <tr>
      <td style="text-align: left;">③ 临时工作区线程私有</td>
      <td style="text-align: left;"><strong>本版不含任何临时工作区</strong>——<code>microkernel_4x4</code> 在寄存器中完成计算，A、B 均为就地读取</td>
      <td style="text-align: left;"><strong>自动满足</strong></td>
    </tr>
  </tbody>
</table>

三条判据均成立，因此**无需 `private` 子句，无需锁，也无需原子操作**。这一便利来源于"C 的各块互不重叠"这一几何性质。

### 💡 关注点：`collapse(2)` 的必要性

以 $1024^3$、块尺寸 64 为例，若仅并行外层 `ii` 循环，任务数为 $M/64 = 16$ 个：

- 8 线程：$16 = 8 \times 2$，尚可均衡；
- 12 线程：$16 = 12 + 4$，第二轮仅有 4 个线程执行计算，其余 8 个处于空闲，并行效率约为 67%；
- 32 线程及以上：半数线程无法分配到任务。

添加 `collapse(2)` 后，`ii` 与 `jj` 被折叠为一个包含 $(M/64)\times(N/64) = 256$ 个任务的迭代空间，负载均衡方成为可能。

> **采用 `schedule(dynamic)` 的理由**：各块的工作量并不完全相等——右边界与下边界的块尺寸较小，且需要执行标量补齐路径。静态划分会导致分配到边界块的线程提前完成而后续等待。动态调度按需领取任务，其代价是每次领取带来的同步开销；本例中每个任务约包含 200 万次浮点运算，该开销可以忽略。

请对照运行结果：`OMP+NEON Tiled` 应为当前 GFLOPS 最高的一行，且 `Check` 为 `PASS`。

In [ ]:
%%writefile src_gemm/omp_gemm_v3.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. The Chapter 3 v4 kernel: 4x4 register blocking + cache blocking, with NO
//    packing step.  The point that matters for this chapter: this routine uses
//    NO scratch buffer at all.  It only reads A and B and writes disjoint
//    blocks of C.  Remember that when you get to v4.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 4. First fusion: OpenMP x the cache-blocked NEON kernel.
//
//    ONE pragma, nothing else changed - and it is correct.  Check the three
//    criteria against this loop nest:
//      (1) no cross-iteration dependence: each (ii, jj) task computes its own
//          C block from scratch;
//      (2) disjoint writes: task (ii, jj) writes only C[ii..ii+63][jj..jj+63];
//      (3) no scratch buffer at all - microkernel_4x4 works in registers and
//          reads A and B in place.
//    All three hold, so nothing needs to be made private and no lock is needed.
//
//    collapse(2) matters here: parallelising ii alone gives only M/64 = 16
//    tasks at 1024^3, which cannot be balanced over more than a handful of
//    threads.  Collapsing gives (M/64) x (N/64) = 256.
// ---------------------------------------------------------
void gemm_omp_neon_tiled(const float* restrict A, const float* restrict B,
                         float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP v3: + OpenMP x Cache-Blocked NEON (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive OpenMP on the scalar triple loop
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_naive(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_omp = (end - start) / NTIMES;
  report("OMP Naive", t_omp, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // OpenMP x cache-blocked NEON, one pragma, no scratch buffer
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_neon_tiled(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("OMP+NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_v3.c", "src_gemm/omp_gemm_v3")
out_v3 = run_bin(BIN, 1024, 1024, 1024, NT)

## 8. v4 · 同一条并行指令用于打包版本

v3 的并行化未遇到任何障碍。下一步将同一条指令应用于第三章的**打包版本**（`gemm_neon_1t`，即 v1 中已经度量过的单线程最优实现），其余代码不作任何修改。

```c
  float* packed_B = (float*)aligned_alloc(16, ...);   // 位于并行域【外部】

#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K, packed_B)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      ...
      pack_matrix_B(cur_K, cur_N, &B[kk*N+jj], N, packed_B);        // 写入
      microkernel_4x4_packed(..., packed_B, ...);                   // 读取
    }
  }
```

### 🐛 运行结果：`Check` 列报告 `FAIL`

**建议先自行分析原因，再阅读下文。** 提示：v3 与 v4 在结构上的唯一差异是什么？

<details>
<summary>参考答案</summary>

判据 ① 与 ② 依然成立——C 的各块仍然互不相交，与 v3 完全一致。**失效的是判据 ③**：打包版本引入了暂存区 `packed_B`，而它在并行域**外部**分配，指针因此属于 `shared`。于是：

- 线程 0 将其负责的 B 块打包写入 `packed_B`；
- 与此同时线程 1 也将另一块完全不同的 B 数据打包写入**同一块内存**；
- 两个线程随后均从该内存读取数据参与乘加运算，读到的具体内容取决于操作系统的调度时序。

</details>

### 💡 该缺陷的三个特征

1. **不产生崩溃**：不存在越界访问与空指针解引用，程序正常结束并输出完整的性能表格；
2. **不产生编译警告**：`gcc -Wall -Wextra -fopenmp` 无任何警告。`default(none)` 同样无法拦截——因为代码中确实**显式声明**了 `packed_B` 为 `shared`，编译器据此认为这是设计意图；
3. **小规模无法复现**：以 `64 64 64` 运行时结果**恒为 `PASS`**，因为此时整个矩阵仅包含 1 个 64×64 的块、即 1 个任务，不存在两个线程同时打包的情形；规模增至 `100 100 100`（4 个块）后才开始出现 `FAIL`。

> **🎓 这三个特征共同构成了并行程序与串行程序的本质差异**：串行程序的缺陷是确定性的，每次运行均可复现；**并行程序的缺陷是概率性的**，其表现取决于线程数、调度顺序、任务粒度乃至系统当前负载。
>
> 因此，**在并行程序中"测试通过"并不等价于"实现正确"**。必须以足够大的问题规模、足够多的线程数反复验证，必要时借助 `valgrind --tool=helgrind` 或 `gcc -fsanitize=thread` 等工具进行检测。

> ⚠️ 若本次运行中 `OMP+Pack Race` 一行显示 `PASS`，请增大线程数或问题规模后重新运行——这一现象恰好印证了上述第 3 条特征。

In [ ]:
%%writefile src_gemm/omp_gemm_v4.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. The Chapter 3 v4 kernel: 4x4 register blocking + cache blocking, with NO
//    packing step.  The point that matters for this chapter: this routine uses
//    NO scratch buffer at all.  It only reads A and B and writes disjoint
//    blocks of C.  Remember that when you get to v4.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 4. First fusion: OpenMP x the cache-blocked NEON kernel.
//
//    ONE pragma, nothing else changed - and it is correct.  Check the three
//    criteria against this loop nest:
//      (1) no cross-iteration dependence: each (ii, jj) task computes its own
//          C block from scratch;
//      (2) disjoint writes: task (ii, jj) writes only C[ii..ii+63][jj..jj+63];
//      (3) no scratch buffer at all - microkernel_4x4 works in registers and
//          reads A and B in place.
//    All three hold, so nothing needs to be made private and no lock is needed.
//
//    collapse(2) matters here: parallelising ii alone gives only M/64 = 16
//    tasks at 1024^3, which cannot be balanced over more than a handful of
//    threads.  Collapsing gives (M/64) x (N/64) = 256.
// ---------------------------------------------------------
void gemm_omp_neon_tiled(const float* restrict A, const float* restrict B,
                         float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 5. [DELIBERATELY BROKEN] The SAME pragma as v3, this time on the PACKED
//    kernel - and now it is wrong.
//
//    Criteria (1) and (2) still hold: the C blocks are disjoint, exactly as in
//    v3.  What changed is criterion (3): gemm_neon_1t owns a scratch buffer,
//    packed_B, and it is allocated ONCE, OUTSIDE the parallel region.  The
//    pointer is therefore shared, so every thread packs its own B block into
//    THE SAME memory and then reads back whatever the last writer left there.
//
//    The result is silently wrong: no crash, no compiler warning, no OpenMP
//    error - just a FAIL in the Check column.
//
//    This function is kept in the benchmark on purpose.  Do not "fix" it here;
//    v5 shows the fix, and the diff is one line.
// ---------------------------------------------------------
void gemm_omp_pack_race(const float* restrict A, const float* restrict B,
                        float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) return;

#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K, packed_B)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);   // RACE
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);        // RACE
      }
    }
  }
  free(packed_B);
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP v4: + OpenMP x PACKED NEON -- HAS A DATA RACE (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive OpenMP on the scalar triple loop
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_naive(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_omp = (end - start) / NTIMES;
  report("OMP Naive", t_omp, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // OpenMP x cache-blocked NEON, one pragma, no scratch buffer
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_neon_tiled(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("OMP+NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // The SAME pragma on the packed kernel -- shared scratch buffer, DATA RACE
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_pack_race(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_race = (end - start) / NTIMES;
  report("OMP+Pack Race", t_race, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_v4.c", "src_gemm/omp_gemm_v4")
out_v4 = run_bin(BIN, 1024, 1024, 1024, NT)

# ⚠️ 预期：最后一行 OMP+Pack Race 的 Check 为 FAIL —— 此为设计结果，非环境故障

## 9. v5 · 线程私有缓冲区

修复方法只有一项：**将缓冲区的分配移入并行域**。

```c
#pragma omp parallel num_threads(threads) \
    default(none) shared(A, B, C, M, N, K, buf_bytes)
  {
    // 在并行域【内部】声明，因而每个线程持有独立的一份
    float* packed_B = (float*)aligned_alloc(16, buf_bytes);

#pragma omp for collapse(2) schedule(dynamic)
    for (int ii = 0; ii < M; ii += BLOCK_M) {
      for (int jj = 0; jj < N; jj += BLOCK_N) {
        ...                                   // 与 v4 完全相同
      }
    }
    free(packed_B);
  }
```

### 💡 关注点：为何必须将 `parallel for` 拆分为 `parallel { for }`

`#pragma omp parallel for` 是 `parallel` 与 `for` 的合并写法，两者之间不能插入任何语句。而此处恰好需要在"创建线程团队"与"分配迭代"之间插入一次 `aligned_alloc`：

<!--
| 写法 | 缓冲区分配次数 | 结果 |
|---|---|---|
| 并行域外 `malloc` + `parallel for` | **1 次**（所有线程共用） | 存在数据竞争（v4） |
| `parallel { malloc; for ... }` | **每线程 1 次** | 正确，且开销可忽略 |
| 将 `malloc` 置于循环体内 | **每个任务 1 次**（$1024^3$ 下为 256 次） | 正确，但内存分配进入热路径 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">写法</th>
      <th style="text-align: left;">缓冲区分配次数</th>
      <th style="text-align: left;">结果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">并行域外 <code>malloc</code> + <code>parallel for</code></td>
      <td style="text-align: left;"><strong>1 次</strong>（所有线程共用）</td>
      <td style="text-align: left;">存在数据竞争（v4）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>parallel { malloc; for ... }</code></td>
      <td style="text-align: left;"><strong>每线程 1 次</strong></td>
      <td style="text-align: left;">正确，且开销可忽略</td>
    </tr>
    <tr>
      <td style="text-align: left;">将 <code>malloc</code> 置于循环体内</td>
      <td style="text-align: left;"><strong>每个任务 1 次</strong>（$1024^3$ 下为 256 次）</td>
      <td style="text-align: left;">正确，但内存分配进入热路径</td>
    </tr>
  </tbody>
</table>

第三种写法同样能够得到正确结果，但把内存分配放进了计算的热路径。**"每线程一次"这一分配粒度，正是 `omp parallel { ... }` 作用域存在的意义。**

### 📌 其余子句保持不变
`collapse(2)` 与 `schedule(dynamic)` 直接继承自 v3，并非问题所在。**v4 与 v5 的实质差异仅在于缓冲区分配语句的位置**，建议实际执行一次 `diff src_omp/omp_gemm_v4.c src_omp/omp_gemm_v5.c` 加以确认。

### ⚠️ 关注点：请比较 v4 与 v5 的耗时

按常理，修复数据竞争属于正确性修改而非性能优化，二者的耗时本应接近。**但实测中 v4 往往明显慢于 v5**，请分析其原因。

<details>
<summary>参考答案</summary>

v4 中所有线程反复写入同一块约 16 KiB 的缓冲区。该缓冲区所在的缓存行在各个核心的私有缓存之间被反复申请为独占状态、再被其他核心夺走，形成**缓存行在核间的持续迁移**（真共享，true sharing）。核心数越多，迁移越频繁，访存代价越高。

因此 v4 相对 v5 不仅结果错误，性能通常也更差。**但这只是本例的具体表现，不能推广为"结果错误的程序必然更慢"**——数据竞争完全可能既产生错误结果、又具有很高的性能表现。判断顺序必须始终是先看 `Check` 列，再看性能数值。

</details>

> **🎓 三个版本的小结**：v3 说明"C 的各块互不重叠"使并行化几乎不需要额外代价；v4 说明**核函数一旦持有内部状态（暂存区、静态变量、全局计数器），这一便利立即失效**；v5 说明修复的代价其实很小——**困难之处不在于修复，而在于发现**。

In [ ]:
%%writefile src_gemm/omp_gemm_v5.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. The Chapter 3 v4 kernel: 4x4 register blocking + cache blocking, with NO
//    packing step.  The point that matters for this chapter: this routine uses
//    NO scratch buffer at all.  It only reads A and B and writes disjoint
//    blocks of C.  Remember that when you get to v4.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 4. First fusion: OpenMP x the cache-blocked NEON kernel.
//
//    ONE pragma, nothing else changed - and it is correct.  Check the three
//    criteria against this loop nest:
//      (1) no cross-iteration dependence: each (ii, jj) task computes its own
//          C block from scratch;
//      (2) disjoint writes: task (ii, jj) writes only C[ii..ii+63][jj..jj+63];
//      (3) no scratch buffer at all - microkernel_4x4 works in registers and
//          reads A and B in place.
//    All three hold, so nothing needs to be made private and no lock is needed.
//
//    collapse(2) matters here: parallelising ii alone gives only M/64 = 16
//    tasks at 1024^3, which cannot be balanced over more than a handful of
//    threads.  Collapsing gives (M/64) x (N/64) = 256.
// ---------------------------------------------------------
void gemm_omp_neon_tiled(const float* restrict A, const float* restrict B,
                         float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 5. [DELIBERATELY BROKEN] The SAME pragma as v3, this time on the PACKED
//    kernel - and now it is wrong.
//
//    Criteria (1) and (2) still hold: the C blocks are disjoint, exactly as in
//    v3.  What changed is criterion (3): gemm_neon_1t owns a scratch buffer,
//    packed_B, and it is allocated ONCE, OUTSIDE the parallel region.  The
//    pointer is therefore shared, so every thread packs its own B block into
//    THE SAME memory and then reads back whatever the last writer left there.
//
//    The result is silently wrong: no crash, no compiler warning, no OpenMP
//    error - just a FAIL in the Check column.
//
//    This function is kept in the benchmark on purpose.  Do not "fix" it here;
//    v5 shows the fix, and the diff is one line.
// ---------------------------------------------------------
void gemm_omp_pack_race(const float* restrict A, const float* restrict B,
                        float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) return;

#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K, packed_B)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);   // RACE
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);        // RACE
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 6. OpenMP x the packed NEON kernel, done right.
//
//    Exactly ONE thing changes with respect to v4: the packed buffer is
//    allocated INSIDE the parallel region, so every thread owns one.  That
//    requires splitting "omp parallel for" into "omp parallel { omp for }",
//    because the allocation must happen once per thread, not once per task.
//
//    collapse(2) and schedule(dynamic) are carried over from v3 unchanged -
//    they were never the problem.
// ---------------------------------------------------------
void gemm_omp_pack(const float* restrict A, const float* restrict B,
                   float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  size_t buf_bytes = (size_t)BLOCK_K * ldb_pack * sizeof(float);

#pragma omp parallel num_threads(threads) \
    default(none) shared(A, B, C, M, N, K, buf_bytes)
  {
    // Declared inside the parallel region => one private buffer per thread.
    float* packed_B = (float*)aligned_alloc(16, buf_bytes);

    if (packed_B) {
#pragma omp for collapse(2) schedule(dynamic)
      for (int ii = 0; ii < M; ii += BLOCK_M) {
        for (int jj = 0; jj < N; jj += BLOCK_N) {
          int cur_M = MIN(BLOCK_M, M - ii);
          int cur_N = MIN(BLOCK_N, N - jj);

          // Threads write to disjoint (ii, jj) blocks of C: no race, no lock.
          for (int r = 0; r < cur_M; r++)
            memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

          for (int kk = 0; kk < K; kk += BLOCK_K) {
            int cur_K = MIN(BLOCK_K, K - kk);
            pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
            microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                                   packed_B, &C[ii * N + jj], N);
          }
        }
      }
      free(packed_B);
    }
  }
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP v5: + Per-Thread Packing Buffer -- the Fix (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive OpenMP on the scalar triple loop
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_naive(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_omp = (end - start) / NTIMES;
  report("OMP Naive", t_omp, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // OpenMP x cache-blocked NEON, one pragma, no scratch buffer
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_neon_tiled(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("OMP+NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // The SAME pragma on the packed kernel -- shared scratch buffer, DATA RACE
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_pack_race(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_race = (end - start) / NTIMES;
  report("OMP+Pack Race", t_race, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Same kernel, buffer moved inside the parallel region -- one line, fixed
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_pack(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_pack = (end - start) / NTIMES;
  report("OMP+Pack", t_pack, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_v5.c", "src_gemm/omp_gemm_v5")
out_v5 = run_bin(BIN, 1024, 1024, 1024, NT)

## 10. 📈 性能可视化（基于 v5 的六版本结果）

v5 的输出包含全部六个实现，据此绘制**加速比**与 **GFLOPS** 两张柱状图。

- 灰色表示无明显加速，蓝色表示存在加速，**红色标示通过校验的实现中性能最优者**；
- **橙色带斜纹表示 `Check` 未通过**（`OMP+Pack Race`）。

> 📌 图中特意将未通过校验的实现单独着色。请注意：**该柱子的高度不具有任何参考价值**——无论它在图中偏高还是偏低，一个计算结果错误的实现都不参与性能比较。

In [ ]:
rows_v5 = parse_table(out_v5)
chk_v5 = parse_check(out_v5)
for r in rows_v5:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:8.2f}x   {chk_v5.get(r["method"], ""):4s}')
plot_speedup(rows_v5, f"OpenMP x NEON GEMM: speedup (1024^3, {NT} threads)", chk_v5)
plot_gflops(parse_gflops(out_v5), f"OpenMP x NEON GEMM: GFLOPS (1024^3, {NT} threads)", chk_v5)

## 11. 🔬 强扩展性研究

前面五个版本构成"修改代码"的递进。本节转入另一个维度：**代码保持不变，仅改变线程数**，考察性能随核心数的变化规律。

### 11.1 两类扩展性

<!--
| 类型 | 问题规模 | 考察的问题 | 本节涉及 |
|---|---|---|---|
| **强扩展（Strong scaling）** | **固定** | 对同一问题，增加核心数能够缩短多少时间 | 是 |
| 弱扩展（Weak scaling） | 随线程数**成比例增大** | 在每核负载不变的前提下，能否保持总耗时不变 | 动手练习 5 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">类型</th>
      <th style="text-align: left;">问题规模</th>
      <th style="text-align: left;">考察的问题</th>
      <th style="text-align: left;">本节涉及</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>强扩展（Strong scaling）</strong></td>
      <td style="text-align: left;"><strong>固定</strong></td>
      <td style="text-align: left;">对同一问题，增加核心数能够缩短多少时间</td>
      <td style="text-align: left;">是</td>
    </tr>
    <tr>
      <td style="text-align: left;">弱扩展（Weak scaling）</td>
      <td style="text-align: left;">随线程数<strong>成比例增大</strong></td>
      <td style="text-align: left;">在每核负载不变的前提下，能否保持总耗时不变</td>
      <td style="text-align: left;">动手练习 5</td>
    </tr>
  </tbody>
</table>

$$S(T)=\frac{T_1}{T_T}\quad(\text{加速比})\qquad\qquad E(T)=\frac{S(T)}{T}\times 100\%\quad(\text{并行效率})$$

### 11.2 ⚠️ 基准必须取"同一份代码的单线程运行"

这是本节最容易出错之处。$T_1$ **必须**取 `gemm_omp_pack` 在 1 线程下的耗时，**不能**取串行标量基准——否则会将 SIMD 优化的收益一并计入"并行加速比"，得到远超线程数的加速比与远超 100% 的"并行效率"，结论没有意义。

> 换言之，第 6 节所讨论的加速比局限性，在扩展性研究中正是通过"固定代码、仅变线程数"来规避的。**这也是加速比唯一严格可信的用法。**

### 11.3 规模选取

线程数较多时，$1024^3$ 的单次执行仅需数毫秒，OpenMP 的线程团队创建与调度开销以及测量噪声将掩盖有效信号。本节采用 $2048^3$（计算量为 8 倍），使曲线末端保持清晰。本节**不执行串行基准**（$2048^3$ 的标量三重循环单次耗时可达数十秒）。

### 11.4 分析方法

运行完成后，请按以下三个方向解读曲线形状：

<!--
| 观察到的现象 | 可能的原因 | 应进一步检查的内容 |
|---|---|---|
| 并行效率在较小线程数处即出现下降 | 任务粒度或负载均衡存在问题 | 任务总数是否充足（$1024^3$、块尺寸 64 时仅有 256 个）；更换 `schedule` 策略 |
| 效率曲线在某个线程数处出现**拐点** | 跨越了某一硬件边界 | 是否跨越 NUMA 节点；是否开始使用 SMT 逻辑核 |
| 效率整体平缓下降 | 内存带宽或共享 Cache 争用 | 按下式估算带宽需求，与本机内存带宽比较 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">观察到的现象</th>
      <th style="text-align: left;">可能的原因</th>
      <th style="text-align: left;">应进一步检查的内容</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">并行效率在较小线程数处即出现下降</td>
      <td style="text-align: left;">任务粒度或负载均衡存在问题</td>
      <td style="text-align: left;">任务总数是否充足（$1024^3$、块尺寸 64 时仅有 256 个）；更换 <code>schedule</code> 策略</td>
    </tr>
    <tr>
      <td style="text-align: left;">效率曲线在某个线程数处出现<strong>拐点</strong></td>
      <td style="text-align: left;">跨越了某一硬件边界</td>
      <td style="text-align: left;">是否跨越 NUMA 节点；是否开始使用 SMT 逻辑核</td>
    </tr>
    <tr>
      <td style="text-align: left;">效率整体平缓下降</td>
      <td style="text-align: left;">内存带宽或共享 Cache 争用</td>
      <td style="text-align: left;">按下式估算带宽需求，与本机内存带宽比较</td>
    </tr>
  </tbody>
</table>

**带宽需求的估算方法**：本实现在块尺寸 64 下，A 被读取 $N/64$ 遍、B 被读取 $M/64$ 遍、C 写入一遍，因此一次 GEMM 调用的访存总量约为

$$\text{Bytes} \approx \left(\frac{N}{64}\cdot MK + \frac{M}{64}\cdot KN + MN\right)\times 4$$

将其除以本次测得的耗时，即可得到实际带宽需求，再与 `lscpu`／厂商手册给出的内存带宽比较。**若估算值远低于本机带宽，则效率下降的原因不在访存**，应转而检查任务粒度、NUMA 或线程绑定。

In [ ]:
%%writefile src_gemm/omp_gemm_scale.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. The Chapter 3 v4 kernel: 4x4 register blocking + cache blocking, with NO
//    packing step.  The point that matters for this chapter: this routine uses
//    NO scratch buffer at all.  It only reads A and B and writes disjoint
//    blocks of C.  Remember that when you get to v4.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 4. First fusion: OpenMP x the cache-blocked NEON kernel.
//
//    ONE pragma, nothing else changed - and it is correct.  Check the three
//    criteria against this loop nest:
//      (1) no cross-iteration dependence: each (ii, jj) task computes its own
//          C block from scratch;
//      (2) disjoint writes: task (ii, jj) writes only C[ii..ii+63][jj..jj+63];
//      (3) no scratch buffer at all - microkernel_4x4 works in registers and
//          reads A and B in place.
//    All three hold, so nothing needs to be made private and no lock is needed.
//
//    collapse(2) matters here: parallelising ii alone gives only M/64 = 16
//    tasks at 1024^3, which cannot be balanced over more than a handful of
//    threads.  Collapsing gives (M/64) x (N/64) = 256.
// ---------------------------------------------------------
void gemm_omp_neon_tiled(const float* restrict A, const float* restrict B,
                         float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 5. [DELIBERATELY BROKEN] The SAME pragma as v3, this time on the PACKED
//    kernel - and now it is wrong.
//
//    Criteria (1) and (2) still hold: the C blocks are disjoint, exactly as in
//    v3.  What changed is criterion (3): gemm_neon_1t owns a scratch buffer,
//    packed_B, and it is allocated ONCE, OUTSIDE the parallel region.  The
//    pointer is therefore shared, so every thread packs its own B block into
//    THE SAME memory and then reads back whatever the last writer left there.
//
//    The result is silently wrong: no crash, no compiler warning, no OpenMP
//    error - just a FAIL in the Check column.
//
//    This function is kept in the benchmark on purpose.  Do not "fix" it here;
//    v5 shows the fix, and the diff is one line.
// ---------------------------------------------------------
void gemm_omp_pack_race(const float* restrict A, const float* restrict B,
                        float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) return;

#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K, packed_B)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);   // RACE
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);        // RACE
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 6. OpenMP x the packed NEON kernel, done right.
//
//    Exactly ONE thing changes with respect to v4: the packed buffer is
//    allocated INSIDE the parallel region, so every thread owns one.  That
//    requires splitting "omp parallel for" into "omp parallel { omp for }",
//    because the allocation must happen once per thread, not once per task.
//
//    collapse(2) and schedule(dynamic) are carried over from v3 unchanged -
//    they were never the problem.
// ---------------------------------------------------------
void gemm_omp_pack(const float* restrict A, const float* restrict B,
                   float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  size_t buf_bytes = (size_t)BLOCK_K * ldb_pack * sizeof(float);

#pragma omp parallel num_threads(threads) \
    default(none) shared(A, B, C, M, N, K, buf_bytes)
  {
    // Declared inside the parallel region => one private buffer per thread.
    float* packed_B = (float*)aligned_alloc(16, buf_bytes);

    if (packed_B) {
#pragma omp for collapse(2) schedule(dynamic)
      for (int ii = 0; ii < M; ii += BLOCK_M) {
        for (int jj = 0; jj < N; jj += BLOCK_N) {
          int cur_M = MIN(BLOCK_M, M - ii);
          int cur_N = MIN(BLOCK_N, N - jj);

          // Threads write to disjoint (ii, jj) blocks of C: no race, no lock.
          for (int r = 0; r < cur_M; r++)
            memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

          for (int kk = 0; kk < K; kk += BLOCK_K) {
            int cur_K = MIN(BLOCK_K, K - kk);
            pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
            microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                                   packed_B, &C[ii * N + jj], N);
          }
        }
      }
      free(packed_B);
    }
  }
}

// ---------------------------------------------------------
// Strong-scaling driver
//   Runs ONLY gemm_omp_neon, sweeping the thread count.
//   The 1-thread run of the SAME code is the reference for both correctness
//   and speedup - that is what "strong scaling" means.  Comparing against the
//   scalar baseline instead would silently fold the SIMD gain into the
//   parallel speedup and make the efficiency numbers meaningless.
// ---------------------------------------------------------
int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [max_threads]\n", argv[0]);
    printf("Example: %s 2048 2048 2048\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int max_t = (argc == 5) ? atoi(argv[4]) : 0;
  if (max_t <= 0) max_t = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  // Thread counts to test: 1, 2, 4, 8, ... plus max_t itself
  int list[32];
  int n_list = 0;
  for (int t = 1; t <= max_t && n_list < 30; t *= 2) list[n_list++] = t;
  if (n_list == 0 || list[n_list - 1] != max_t) list[n_list++] = max_t;

  printf("============================================================\n");
  printf(" OMP scaling study: gemm_omp_pack (C = A * B)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: 1 .. %d   (cores visible to OpenMP: %d)\n", max_t,
         omp_get_max_threads());
  printf(" Loops  : %d   (no serial baseline: speedup is relative to 1 thread)\n",
         NTIMES);
  printf("============================================================\n");

  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);
  float* C_test = (float*)aligned_alloc(16, bytes_C);
  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Reference = the 1-thread run of the same kernel (also warms up the pages)
  gemm_omp_pack(A, B, C_ref, M, N, K, 1);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double t1 = 0.0;

  printf("\n--------------------------------------------------------------\n");
  printf("| Threads         | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  for (int idx = 0; idx < n_list; idx++) {
    int t = list[idx];

    // One untimed warm-up so that team creation and first-touch are not
    // charged to the measurement
    gemm_omp_pack(A, B, C_test, M, N, K, t);

    memset(C_test, 0, bytes_C);
    double start = get_time_ms();
    for (int r = 0; r < NTIMES; r++) gemm_omp_pack(A, B, C_test, M, N, K, t);
    double end = get_time_ms();
    double tt = (end - start) / NTIMES;
    if (idx == 0) t1 = tt;

    char name[16];
    snprintf(name, sizeof(name), "%d", t);
    report(name, tt, t1, ops, check_result(C_ref, C_test, M, N, K));
  }

  printf("--------------------------------------------------------------\n");
  printf("\nParallel efficiency = Speedup / Threads.  A value that falls off\n");
  printf("as threads grow tells you WHERE the scaling stops, not THAT it stops.\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_scale.c", "src_gemm/omp_gemm_scale")
out_scale = run_bin(BIN, 2048, 2048, 2048, NT)

rows_scale = parse_table(out_scale)
plot_scaling(rows_scale, f"Strong scaling of gemm_omp_pack (2048^3, up to {NT} threads)")

## 12. 结果分析

> 注：具体数值随硬件平台、核心数、问题规模、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

---

**① `OMP Naive` 的加速比数值可观，但其绝对性能低于单线程 NEON 实现。**

请用本次运行的数据填写下表并作出判断：

<!--
| 需要填写的量 | 取值来源 | 判断依据 |
|---|---|---|
| `NEON 1-Thread` 的 GFLOPS | v1 表格 | **单线程**的性能水平 |
| `OMP Naive` 的 GFLOPS | v2 表格 | 占用全部核心后的性能水平 |
| `OMP Naive` 的并行效率 | 加速比 ÷ 线程数 | 与后续 v3、v5 的并行效率比较 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">需要填写的量</th>
      <th style="text-align: left;">取值来源</th>
      <th style="text-align: left;">判断依据</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>NEON 1-Thread</code> 的 GFLOPS</td>
      <td style="text-align: left;">v1 表格</td>
      <td style="text-align: left;"><strong>单线程</strong>的性能水平</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>OMP Naive</code> 的 GFLOPS</td>
      <td style="text-align: left;">v2 表格</td>
      <td style="text-align: left;">占用全部核心后的性能水平</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>OMP Naive</code> 的并行效率</td>
      <td style="text-align: left;">加速比 ÷ 线程数</td>
      <td style="text-align: left;">与后续 v3、v5 的并行效率比较</td>
    </tr>
  </tbody>
</table>

通常可以观察到两个现象：其一，`OMP Naive` 的 GFLOPS **低于** `NEON 1-Thread`，即占用全部核心的并行程序性能不及单线程程序；其二，`OMP Naive` 的并行效率**明显低于** v3、v5。

后一现象说明：朴素三重循环属于**访存受限**的实现（`B[k*N+j]` 的访问步长为 $N$，缓存行利用率极低），多个线程同时以低效方式访问内存，会迅速耗尽存储系统的供给能力。**因此低效实现付出的是双重代价：绝对性能低，且扩展性同样受限。**

> **🎓 由此得到本实验的第一条核心结论**：加速比只能反映"相对自身串行版本的改善程度"，不能反映实现质量。**优化的正确顺序是先单核、后多核**——单核的每一分收益都会被线程数放大，单核的每一分浪费同样会被放大 $T$ 倍。

---

**② v3 仅添加一条并行指令即通过校验，其原因在于 GEMM 的输出天然可划分。**

C 的每个 $(ii, jj)$ 块由一个任务独占，块与块严格不相交。这一几何性质使判据 ① ② 自动成立，因而既不需要 `reduction`，也不需要 `critical`。

**并非所有算法都具备这一性质**：直方图统计、稀疏矩阵累加、图遍历等在这一步都会遇到写冲突。这也解释了本实验为何始终不并行 `k` 循环——`k` 是规约方向，所有 $k$ 累加到同一个 $C_{ij}$，判据 ① 直接失效。

---

**③ v4 的 `FAIL` 与 v3 的 `PASS`，差异仅在于核函数是否持有内部状态。**

两个版本的并行指令逐字相同，对 C 的写入模式也相同，唯一的变化是打包版本持有 `packed_B` 暂存区。**判据 ③ 正是为这一情形设立的**，而它恰恰是三条判据中最易被忽略的一条。

请特别记录本次运行中该缺陷的三个特征：不产生崩溃、不产生编译警告、以 `64 64 64` 运行时无法复现。

---

**④ v4 与 v5 的耗时差异，反映的是缓存行在核间的迁移代价。**

修复数据竞争属于正确性修改，理论上不改变计算量：每线程额外分配一块约 16 KiB 的缓冲区，开销可以忽略。但实测中 v4 往往**明显慢于** v5，原因是 v4 的所有线程反复写入同一块缓冲区，该缓冲区所在的缓存行需要在各核心的私有缓存之间反复迁移（真共享）。核心数越多，这一代价越显著。

> ⚠️ **不可由此推出"结果错误的实现必然更慢"**。数据竞争完全可能既产生错误结果、又具有很高的性能表现。**判断顺序必须始终是先看 `Check` 列，再看性能数值。**

---

**⑤ v5 相对 v3 的领先，来源于第三章的打包优化，而非 OpenMP。**

两者的并行结构完全相同（同样的 `collapse(2)`、同样的 `schedule`、同样的任务划分），差异仅在于微内核是否使用打包后的 B。请计算二者 GFLOPS 之比，并与**单线程条件下**打包版本相对 Cache 分块版本的比值对照。

若两个比值接近，说明打包优化的收益在多线程下得以保持；若多线程下的比值明显偏离，则说明打包在并行环境中产生了额外效应，例如每线程的打包缓冲区常驻各自的私有缓存（有利），或多份缓冲区加重了共享 Cache 的压力（不利）。二者均为值得进一步分析的结果。

---

**⑥ 主表的并行效率与扩展性研究的并行效率通常并不相同。**

请对比两组数据：

- 主表（$1024^3$）：用 `OMP+Pack` 的 GFLOPS 除以 `NEON 1-Thread` 的 GFLOPS，得到线程级加速比，再除以线程数得到并行效率；
- 扩展性研究（$2048^3$）：读取满线程数对应的并行效率。

通常**后者高于前者**。原因是 $1024^3$、块尺寸 64 时任务总数仅 256 个，每个任务的工作量较小，线程调度与同步开销所占比重更高；而 $2048^3$ 的任务数为 1024 个，单个任务的工作量也大 4 倍。

> **🎓 由此得到一条重要的方法论**：**并行效率不是实现的固有属性，而是"实现 + 问题规模 + 线程数"三者共同决定的**。报告并行效率时必须同时给出问题规模与线程数，否则该数值无法解释。

---

**⑦ 强扩展性曲线的形状比其终点更具信息量。**

请按第 11.4 节的三个方向判断本机曲线属于哪一类，并给出机制性解释。特别提醒：在得出"受内存带宽制约"的结论之前，**务必先用 11.4 节给出的公式估算实际带宽需求**，再与本机内存带宽比较——分块良好的 GEMM 访存量很低，效率下降往往另有原因。

> 📏 **先度量噪声，再解释差异。** 本 Notebook 的五个版本单元格各自重新测量了一遍此前的各个实现，因此可以直接统计同一实现在多次运行中的耗时极差，作为噪声水平的参考。请注意：**耗时最短的实现，其相对抖动通常最大**，因为固定开销在总耗时中所占比重更高。凡小于噪声水平的差异，一律不作解释。

---

### 🎓 结论

本实验将三个并行层级串联为完整的一条链：

<!--
| 层级 | 手段 | 作用范围 | 在本实验中的角色 |
|---|---|---|---|
| DLP（第三章） | NEON 向量化 + 寄存器/Cache 分块 + 内存打包 | 单核 | **基准**，而非终点 |
| TLP（本章） | OpenMP 划分 `(ii, jj)` 任务 | 多核 | 本章主体 |
| 合计 | 两者**相乘** | — | v5 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">层级</th>
      <th style="text-align: left;">手段</th>
      <th style="text-align: left;">作用范围</th>
      <th style="text-align: left;">在本实验中的角色</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">DLP（第三章）</td>
      <td style="text-align: left;">NEON 向量化 + 寄存器/Cache 分块 + 内存打包</td>
      <td style="text-align: left;">单核</td>
      <td style="text-align: left;"><strong>基准</strong>，而非终点</td>
    </tr>
    <tr>
      <td style="text-align: left;">TLP（本章）</td>
      <td style="text-align: left;">OpenMP 划分 <code>(ii, jj)</code> 任务</td>
      <td style="text-align: left;">多核</td>
      <td style="text-align: left;">本章主体</td>
    </tr>
    <tr>
      <td style="text-align: left;">合计</td>
      <td style="text-align: left;">两者<strong>相乘</strong></td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">v5</td>
    </tr>
  </tbody>
</table>

**三个层级相乘，意味着任何一层的低效都会被其余各层放大。** 这正是工业级 BLAS 库（OpenBLAS、BLIS）总是先将单核微内核优化至接近峰值、再考虑多线程的原因；若顺序颠倒，多核只会将浪费同样放大 $T$ 倍。

v4 的 `FAIL` 则提示了另一件事：**并行引入的首要风险不是性能不足，而是结果错误。** 在串行代码中，"函数内部申请一块临时缓冲区"是极为常见的写法；一旦进入并行域，它就成为共享状态。**审查并行代码时，应当先检查每个函数的内部状态，再检查其并行指令。**

## 13. 🚀 扩展实验：OpenMP × 8×8 双打包微内核

> **本节由学生独立完成。** 8×8 双打包微内核本身是**第三章扩展实验的内容**，此处原样提供，无需重新实现。需要补全的是**并行化部分**：两处 `TODO`。

### 🎯 目标

将第三章算术强度最高的微内核应用于多线程环境，使其成为表中性能最高的实现。

### 🧩 与 v5 的两点差异

1. 需要**两块**打包缓冲区（A 与 B 均需打包），因此每个线程须分配 `packed_A` 与 `packed_B` 两块；
2. 分块尺寸由 64 增大为 **128**（8×8 微内核适配更大的块）。请一并考虑这一改动对**任务总数**的影响，详见下文思考题。

### ✅ 验收标准

1. `Check` 列为 **PASS**；
2. 以 `1023 1023 1023`、`130 133 137` 等非 8 倍数规模验证边界处理，仍为 **PASS**；
3. **在多个线程数下反复验证**（至少 1、2、半数核心、全部核心各一次）——参见第 8 节：数据竞争在任务数不足时不会显现；
4. `OMP+NEON 8x8` 成为表中 GFLOPS 最高的一行。

> 💡 **骨架的初始状态**：两处 `TODO` 未补全时，该实现**可以编译、可以运行，且 `Check` 为 `PASS`**——因为不含任何并行指令时它就是一个单线程实现，正确但未使用多核。需要注意的是，由于其微内核本身优于 v1 中的 4×4 版本，**它的 GFLOPS 可能高于 `NEON 1-Thread`，但必然远低于任何多线程版本**。
>
> 学生的任务是**在不破坏 `PASS` 的前提下提升其性能**。这一设定比"将 FAIL 改为 PASS"更贴近真实的并行开发过程：**代码原本是正确的，是新增的并行化有可能使其出错。**

### 💭 完成后的思考题

- $1024^3$ 下，块尺寸 128 对应 $(1024/128)^2 = 64$ 个任务，而 v5 为 256 个。**在使用全部核心时，本扩展实现的并行效率是否会低于 v5？** 请实测验证，并说明"增大分块以提高算术强度"与"减小分块以改善负载均衡"之间应如何权衡。
- 每个线程现持有两块打包缓冲区（约 $128\times136\times4\times2 \approx 139$ KiB）。$T$ 个线程合计占用多少？与本机 L2/L3 容量比较，是否可能成为新的瓶颈？
- 将 `schedule(dynamic)` 分别改为 `schedule(static)` 与 `schedule(guided)` 各测量一次。任务总数减少后，调度策略的影响是增大还是减小？

### 🔬 附加探究（无需修改代码，仅调整环境变量）

- **线程绑定**：分别以 `OMP_PROC_BIND=false`、`close`、`spread` 运行，比较扩展性曲线。在 NUMA 架构上差异可能较为显著。
- **NUMA first-touch**：本实验中 A、B 由**串行代码初始化**，按 first-touch 策略，所有内存页均分配在执行初始化的那个 NUMA 节点上。请用 `numactl --hardware` 查看本机拓扑，再以 `numactl --interleave=all ./omp_gemm_scale 2048 2048 2048` 重新运行扩展性研究并比较差异。
- **进阶**：为 `main` 中 A、B 的初始化循环添加 `#pragma omp parallel for`，使每个线程预先访问其后续将要使用的内存页，再对比一次。这一做法即高性能计算中的 **first-touch 优化**。

In [ ]:
%%writefile src_gemm/omp_gemm_ext.c
#include <arm_neon.h>
#include <math.h>
#include <omp.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run takes seconds)

// Cache block sizes, unchanged from Chapter 3: one A block + one B block +
// one C block = 3 x 64 x 64 x 4 B = 48 KiB, which fits in L1D.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking (identical to Chapter 3)
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The rounding error of a float32 dot product grows roughly linearly with the
// number of accumulation steps, so a FIXED absolute tolerance would report a
// bogus FAIL once K becomes large.  tol = 2e-8 * K * max|C_ref| tracks that
// growth while still catching real defects.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling (identical to Chapter 3): when a dimension is not a multiple
// of 4, the remaining strip is finished with scalar code.
//   _store overwrites  (C  = sum)
//   _accum accumulates (C += sum): for K-blocked versions
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial baseline (auto-vectorization forced off)
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// The Chapter 3 result we start from: 4x4 register-blocked micro-kernel
// operating on a PACKED copy of the B block.  On a single core this reached
// 27.71 GFLOPS = about 2/3 of the single-core peak, while using only ~5% of
// the available memory bandwidth.  That headroom is exactly what OpenMP is
// going to spend.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad, keeps 16 B align
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 2. Single-thread NEON (Chapter 3 v5, unchanged) - the real starting line
//    NOTE the single packed_B buffer allocated ONCE for the whole call.
//    Perfectly fine with one thread; remember it for v3.
// ---------------------------------------------------------
void gemm_neon_1t(const float* restrict A, const float* restrict B,
                  float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 3. Naive OpenMP: parallelize the i loop of the SCALAR triple loop.
//    Each i is an independent row of C, so there is no race and no reduction.
//    This is the version whose SPEEDUP looks best and whose GFLOPS is worst -
//    the central lesson of this lab.
// ---------------------------------------------------------
void gemm_omp_naive(const float* restrict A, const float* restrict B,
                    float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for num_threads(threads) schedule(static) \
    default(none) shared(A, B, C, M, N, K)
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. The Chapter 3 v4 kernel: 4x4 register blocking + cache blocking, with NO
//    packing step.  The point that matters for this chapter: this routine uses
//    NO scratch buffer at all.  It only reads A and B and writes disjoint
//    blocks of C.  Remember that when you get to v4.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

// ---------------------------------------------------------
// 4. First fusion: OpenMP x the cache-blocked NEON kernel.
//
//    ONE pragma, nothing else changed - and it is correct.  Check the three
//    criteria against this loop nest:
//      (1) no cross-iteration dependence: each (ii, jj) task computes its own
//          C block from scratch;
//      (2) disjoint writes: task (ii, jj) writes only C[ii..ii+63][jj..jj+63];
//      (3) no scratch buffer at all - microkernel_4x4 works in registers and
//          reads A and B in place.
//    All three hold, so nothing needs to be made private and no lock is needed.
//
//    collapse(2) matters here: parallelising ii alone gives only M/64 = 16
//    tasks at 1024^3, which cannot be balanced over more than a handful of
//    threads.  Collapsing gives (M/64) x (N/64) = 256.
// ---------------------------------------------------------
void gemm_omp_neon_tiled(const float* restrict A, const float* restrict B,
                         float* restrict C, int M, int N, int K, int threads) {
#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 5. [DELIBERATELY BROKEN] The SAME pragma as v3, this time on the PACKED
//    kernel - and now it is wrong.
//
//    Criteria (1) and (2) still hold: the C blocks are disjoint, exactly as in
//    v3.  What changed is criterion (3): gemm_neon_1t owns a scratch buffer,
//    packed_B, and it is allocated ONCE, OUTSIDE the parallel region.  The
//    pointer is therefore shared, so every thread packs its own B block into
//    THE SAME memory and then reads back whatever the last writer left there.
//
//    The result is silently wrong: no crash, no compiler warning, no OpenMP
//    error - just a FAIL in the Check column.
//
//    This function is kept in the benchmark on purpose.  Do not "fix" it here;
//    v5 shows the fix, and the diff is one line.
// ---------------------------------------------------------
void gemm_omp_pack_race(const float* restrict A, const float* restrict B,
                        float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) return;

#pragma omp parallel for collapse(2) num_threads(threads) schedule(dynamic) \
    default(none) shared(A, B, C, M, N, K, packed_B)
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_M = MIN(BLOCK_M, M - ii);
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);   // RACE
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);        // RACE
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 6. OpenMP x the packed NEON kernel, done right.
//
//    Exactly ONE thing changes with respect to v4: the packed buffer is
//    allocated INSIDE the parallel region, so every thread owns one.  That
//    requires splitting "omp parallel for" into "omp parallel { omp for }",
//    because the allocation must happen once per thread, not once per task.
//
//    collapse(2) and schedule(dynamic) are carried over from v3 unchanged -
//    they were never the problem.
// ---------------------------------------------------------
void gemm_omp_pack(const float* restrict A, const float* restrict B,
                   float* restrict C, int M, int N, int K, int threads) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  size_t buf_bytes = (size_t)BLOCK_K * ldb_pack * sizeof(float);

#pragma omp parallel num_threads(threads) \
    default(none) shared(A, B, C, M, N, K, buf_bytes)
  {
    // Declared inside the parallel region => one private buffer per thread.
    float* packed_B = (float*)aligned_alloc(16, buf_bytes);

    if (packed_B) {
#pragma omp for collapse(2) schedule(dynamic)
      for (int ii = 0; ii < M; ii += BLOCK_M) {
        for (int jj = 0; jj < N; jj += BLOCK_N) {
          int cur_M = MIN(BLOCK_M, M - ii);
          int cur_N = MIN(BLOCK_N, N - jj);

          // Threads write to disjoint (ii, jj) blocks of C: no race, no lock.
          for (int r = 0; r < cur_M; r++)
            memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

          for (int kk = 0; kk < K; kk += BLOCK_K) {
            int cur_K = MIN(BLOCK_K, K - kk);
            pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
            microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                                   packed_B, &C[ii * N + jj], N);
          }
        }
      }
      free(packed_B);
    }
  }
}

// ---------------------------------------------------------
// 7. [EXTENSION LAB] OpenMP x the 8x8 double-packed micro-kernel
//    The kernel itself is the Chapter 3 extension answer, reproduced here
//    unchanged.  What is new is purely the OpenMP side: BOTH packing buffers
//    are now per-thread, and the (ii, jj) iteration space is what gets
//    distributed.
//    Arithmetic intensity 2.0 FLOP/Byte instead of 1.0.
// ---------------------------------------------------------
#define EXT_BLOCK_M 128
#define EXT_BLOCK_N 128
#define EXT_BLOCK_K 128

// Pack an (M x K) block of A into strips of 8 rows, k-major.
void pack_A_panel_8(int K, int M, const float* A, int lda, float* buffer) {
  for (int i = 0; i < M; i += 8) {
    for (int k = 0; k < K; k++) {
      for (int r = 0; r < 8; r++)
        *buffer++ = (i + r < M) ? A[(i + r) * lda + k] : 0.0f;
    }
  }
}

// Pack a (K x N) block of B into strips of 8 columns, k-major.
void pack_B_panel_8(int K, int N, const float* B, int ldb, float* buffer) {
  for (int j = 0; j < N; j += 8) {
    for (int k = 0; k < K; k++) {
      for (int c = 0; c < 8; c++)
        *buffer++ = (j + c < N) ? B[k * ldb + (j + c)] : 0.0f;
    }
  }
}

// 8x8 micro-kernel: C[0..7][0..7] += A_panel * B_panel
void microkernel_8x8_packed(int K, const float* A_panel, const float* B_panel,
                            float* C, int ldc) {
  float32x4_t c00 = vdupq_n_f32(0.0f), c01 = vdupq_n_f32(0.0f);
  float32x4_t c10 = vdupq_n_f32(0.0f), c11 = vdupq_n_f32(0.0f);
  float32x4_t c20 = vdupq_n_f32(0.0f), c21 = vdupq_n_f32(0.0f);
  float32x4_t c30 = vdupq_n_f32(0.0f), c31 = vdupq_n_f32(0.0f);
  float32x4_t c40 = vdupq_n_f32(0.0f), c41 = vdupq_n_f32(0.0f);
  float32x4_t c50 = vdupq_n_f32(0.0f), c51 = vdupq_n_f32(0.0f);
  float32x4_t c60 = vdupq_n_f32(0.0f), c61 = vdupq_n_f32(0.0f);
  float32x4_t c70 = vdupq_n_f32(0.0f), c71 = vdupq_n_f32(0.0f);

  const float* a_ptr = A_panel;
  const float* b_ptr = B_panel;

  for (int k = 0; k < K; k++) {
    float32x4_t b0 = vld1q_f32(b_ptr);
    float32x4_t b1 = vld1q_f32(b_ptr + 4);
    b_ptr += 8;
    __builtin_prefetch(b_ptr + 64, 0, 3);

    float a0 = *a_ptr++;  c00 = vfmaq_n_f32(c00, b0, a0);  c01 = vfmaq_n_f32(c01, b1, a0);
    float a1 = *a_ptr++;  c10 = vfmaq_n_f32(c10, b0, a1);  c11 = vfmaq_n_f32(c11, b1, a1);
    float a2 = *a_ptr++;  c20 = vfmaq_n_f32(c20, b0, a2);  c21 = vfmaq_n_f32(c21, b1, a2);
    float a3 = *a_ptr++;  c30 = vfmaq_n_f32(c30, b0, a3);  c31 = vfmaq_n_f32(c31, b1, a3);
    float a4 = *a_ptr++;  c40 = vfmaq_n_f32(c40, b0, a4);  c41 = vfmaq_n_f32(c41, b1, a4);
    float a5 = *a_ptr++;  c50 = vfmaq_n_f32(c50, b0, a5);  c51 = vfmaq_n_f32(c51, b1, a5);
    float a6 = *a_ptr++;  c60 = vfmaq_n_f32(c60, b0, a6);  c61 = vfmaq_n_f32(c61, b1, a6);
    float a7 = *a_ptr++;  c70 = vfmaq_n_f32(c70, b0, a7);  c71 = vfmaq_n_f32(c71, b1, a7);
  }

  vst1q_f32(C + 0 * ldc + 0, vaddq_f32(vld1q_f32(C + 0 * ldc + 0), c00));
  vst1q_f32(C + 0 * ldc + 4, vaddq_f32(vld1q_f32(C + 0 * ldc + 4), c01));
  vst1q_f32(C + 1 * ldc + 0, vaddq_f32(vld1q_f32(C + 1 * ldc + 0), c10));
  vst1q_f32(C + 1 * ldc + 4, vaddq_f32(vld1q_f32(C + 1 * ldc + 4), c11));
  vst1q_f32(C + 2 * ldc + 0, vaddq_f32(vld1q_f32(C + 2 * ldc + 0), c20));
  vst1q_f32(C + 2 * ldc + 4, vaddq_f32(vld1q_f32(C + 2 * ldc + 4), c21));
  vst1q_f32(C + 3 * ldc + 0, vaddq_f32(vld1q_f32(C + 3 * ldc + 0), c30));
  vst1q_f32(C + 3 * ldc + 4, vaddq_f32(vld1q_f32(C + 3 * ldc + 4), c31));
  vst1q_f32(C + 4 * ldc + 0, vaddq_f32(vld1q_f32(C + 4 * ldc + 0), c40));
  vst1q_f32(C + 4 * ldc + 4, vaddq_f32(vld1q_f32(C + 4 * ldc + 4), c41));
  vst1q_f32(C + 5 * ldc + 0, vaddq_f32(vld1q_f32(C + 5 * ldc + 0), c50));
  vst1q_f32(C + 5 * ldc + 4, vaddq_f32(vld1q_f32(C + 5 * ldc + 4), c51));
  vst1q_f32(C + 6 * ldc + 0, vaddq_f32(vld1q_f32(C + 6 * ldc + 0), c60));
  vst1q_f32(C + 6 * ldc + 4, vaddq_f32(vld1q_f32(C + 6 * ldc + 4), c61));
  vst1q_f32(C + 7 * ldc + 0, vaddq_f32(vld1q_f32(C + 7 * ldc + 0), c70));
  vst1q_f32(C + 7 * ldc + 4, vaddq_f32(vld1q_f32(C + 7 * ldc + 4), c71));
}

void gemm_omp_pack_8x8(const float* restrict A, const float* restrict B,
                       float* restrict C, int M, int N, int K, int threads) {
  size_t szA = (size_t)EXT_BLOCK_K * (EXT_BLOCK_M + 8) * sizeof(float);
  size_t szB = (size_t)EXT_BLOCK_K * (EXT_BLOCK_N + 8) * sizeof(float);
  (void)threads;  // remove this line once TODO 1 actually uses it

  // ============================ TODO 1 ============================
  // Open a parallel region here with num_threads(threads).
  // Both packing buffers must be PER THREAD - remember what v3 showed when a
  // single buffer was shared.  Declaring them inside the parallel region is
  // all it takes.
  //     float* packed_A = (float*)aligned_alloc(16, szA);
  //     float* packed_B = (float*)aligned_alloc(16, szB);
  // ================================================================
  {
    float* packed_A = (float*)aligned_alloc(16, szA);   // <- currently SHARED
    float* packed_B = (float*)aligned_alloc(16, szB);   // <- currently SHARED

    if (packed_A && packed_B) {
  // ============================ TODO 2 ============================
  // Distribute the (ii, jj) iteration space across the threads.
  // Think about three things:
  //   - which loops to fuse so that there are enough independent tasks
  //   - which schedule to use, and why the block workloads are not identical
  //   - whether anything here needs a critical section or an atomic (hint: look
  //     at which elements of C each task writes)
  // ================================================================
      for (int ii = 0; ii < M; ii += EXT_BLOCK_M) {
        for (int jj = 0; jj < N; jj += EXT_BLOCK_N) {
          int cur_M = MIN(EXT_BLOCK_M, M - ii);
          int cur_N = MIN(EXT_BLOCK_N, N - jj);

          for (int r = 0; r < cur_M; r++)
            memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

          for (int kk = 0; kk < K; kk += EXT_BLOCK_K) {
            int cur_K = MIN(EXT_BLOCK_K, K - kk);

            pack_A_panel_8(cur_K, cur_M, &A[ii * K + kk], K, packed_A);
            pack_B_panel_8(cur_K, cur_N, &B[kk * N + jj], N, packed_B);

            for (int i = 0; i < cur_M; i += 8) {
              const float* ap = packed_A + (size_t)i * cur_K;
              for (int j = 0; j < cur_N; j += 8) {
                const float* bp = packed_B + (size_t)j * cur_K;
                if (i + 8 <= cur_M && j + 8 <= cur_N) {
                  microkernel_8x8_packed(cur_K, ap, bp,
                                         &C[(ii + i) * N + (jj + j)], N);
                } else {
                  for (int r = 0; r < 8 && i + r < cur_M; r++) {
                    for (int c = 0; c < 8 && j + c < cur_N; c++) {
                      float sum = 0.0f;
                      for (int k = 0; k < cur_K; k++)
                        sum += ap[k * 8 + r] * bp[k * 8 + c];
                      C[(ii + i + r) * N + (jj + j + c)] += sum;
                    }
                  }
                }
              }
            }
          }
        }
      }
    }
    free(packed_A);
    free(packed_B);
  }
}

int main(int argc, char** argv) {
  if (argc != 4 && argc != 5) {
    printf("Usage: %s <M> <N> <K> [threads]\n", argv[0]);
    printf("Example: %s 1024 1024 1024 8   (0 or omitted = all cores)\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  int threads = (argc == 5) ? atoi(argv[4]) : 0;
  if (threads <= 0) threads = omp_get_max_threads();
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" OMP ext: + 8x8 Double-Packed x OpenMP (STUDENT VERSION)\n");
  printf(" Matrix : A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Threads: %d   (cores visible to OpenMP: %d)\n", threads,
         omp_get_max_threads());
  printf(" Loops  : %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Single-thread NEON, the Chapter 3 result
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_1t(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_1t = (end - start) / NTIMES;
  report("NEON 1-Thread", t_1t, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive OpenMP on the scalar triple loop
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_naive(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_omp = (end - start) / NTIMES;
  report("OMP Naive", t_omp, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // OpenMP x cache-blocked NEON, one pragma, no scratch buffer
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_neon_tiled(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("OMP+NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // The SAME pragma on the packed kernel -- shared scratch buffer, DATA RACE
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_pack_race(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_race = (end - start) / NTIMES;
  report("OMP+Pack Race", t_race, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Same kernel, buffer moved inside the parallel region -- one line, fixed
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_omp_pack(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_pack = (end - start) / NTIMES;
  report("OMP+Pack", t_pack, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // [EXTENSION LAB] OpenMP x 8x8 double-packed micro-kernel
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++)
    gemm_omp_pack_8x8(A, B, C_test, M, N, K, threads);
  end = get_time_ms();
  double t_8x8 = (end - start) / NTIMES;
  report("OMP+NEON 8x8", t_8x8, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/omp_gemm_ext.c", "src_gemm/omp_gemm_ext")
out_ext = run_bin(BIN, 1024, 1024, 1024, NT)

rows_ext = parse_table(out_ext)
if rows_ext:
    chk_ext = parse_check(out_ext)
    plot_gflops(parse_gflops(out_ext), f"With the 8x8 double-packed kernel (1024^3, {NT} threads)", chk_ext)

## 14. 🔧 动手练习

请修改代码或调整环境变量后重新编译运行，观察性能与正确性的变化（建议先独立完成，再阅读思考题）：

1. **度量噪声。** 将 v5 单元格连续运行 3–5 次，记录各实现耗时的极差。多线程会引入额外的抖动来源（线程调度、线程迁移、其他进程占用）。请特别关注一个现象：**耗时最短的实现，其相对抖动往往最大**，并解释原因。此后凡小于噪声水平的差异，一律不作解释。

2. **使数据竞争先消失、再复现。** 以 `64 64 64` 运行 v5，`OMP+Pack Race` 一行将显示 `PASS`。请解释原因，并找出**能够稳定复现 `FAIL` 的最小规模**。随后固定规模为 `1024 1024 1024`，将线程数从 1 逐步增加至全部核心，记录从哪个线程数开始出现 `FAIL`。

3. **复现数据竞争。** 将 v5 中 `float* packed_B = ...` 一行从 `#pragma omp parallel` 的大括号**内部移至外部**，重新编译运行，观察 `Check` 的变化。这即是 v4 与 v5 的全部差异，请用 `diff src_omp/omp_gemm_v4.c src_omp/omp_gemm_v5.c` 加以确认。

4. **考察 `collapse(2)` 的作用。** 删除 v5 中的 `collapse(2)`（仅并行 `ii` 循环），在**不同线程数**下重新测量。$1024^3$ 时仅并行 `ii` 只有 16 个任务，请先预测再验证：线程数为 4、8、12、16 时并行效率各为多少？在哪个线程数处下降最为显著？原因是什么？

5. **调度策略与弱扩展。**（a）将 v5 的 `schedule(dynamic)` 依次改为 `static`、`guided`、`dynamic,4`，比较性能；（b）完成一次**弱扩展**测量：线程数为 $T$ 时取规模 $N = 1024\sqrt[3]{T}$（如 1 线程取 1024、8 线程取 2048），考察总耗时能否保持不变。

6. **分块尺寸与线程数的协同。** 将 v5 的 `BLOCK_M/N/K` 由 64 依次改为 32、96、128，在全部核心下各测量一次。分块减小则任务增多、负载更均衡，但每块的算术强度下降、打包开销占比上升。最优取值是多少？它与单线程条件下的最优取值是否相同？

7. **【进阶】NUMA 与 first-touch。** 先用 `numactl --hardware` 查看本机 NUMA 拓扑，随后：（a）以 `numactl --interleave=all` 重新运行第 11 节的扩展性研究；（b）为 `main` 中 A、B 的初始化循环添加 `#pragma omp parallel for`（注意划分方式应与计算阶段一致），再运行一次。哪一种改善更为明显？

8. **【进阶】线程绑定。** 分别以 `OMP_PROC_BIND=false`、`close`、`spread` 运行扩展性研究，将三条曲线绘制于同一张图中，并结合 `lscpu` 给出的 NUMA 与 SMT 信息解释差异。

9. **【进阶】使用工具检测数据竞争。** 以 `gcc -fsanitize=thread -fopenmp -O1` 重新编译 v4（ThreadSanitizer 要求优化级别不高于 `-O1`），以小规模（如 `256 256 256`）运行，考察其能否指出 `packed_B` 上的竞争；随后对 v5 运行一次作为对照。⚠️ ThreadSanitizer 会使程序减速 5–15 倍，务必使用小规模。

10. **【进阶】与工业实现对比。** 若系统已安装多线程版 OpenBLAS，编写一个调用 `cblas_sgemm` 的程序，在相同规模与线程数下进行对比，分析差距的来源。

## 15. 🤔 思考题

- 为什么说三个并行层级（ILP / DLP / TLP）是**相乘**而非相加的关系？若单核实现仅达到峰值的 1%，将其扩展到 64 个核心后能达到峰值的百分之几？
- `OMP Naive` 取得了数倍的加速比，其 GFLOPS 却低于单线程 NEON 实现。请给出一个**不依赖加速比也能正确排序**的评价指标，并说明加速比在何种条件下才是可信的。
- 访存受限的实现为何不仅绝对性能低，**扩展性同样受限**？请从缓存行利用率与存储系统供给能力两个方面说明。
- v3 与 v4 使用了**逐字相同**的并行指令，为何一个正确、一个错误？请用第 3 节的三条判据作答，并说明 `default(none)` 为何无法拦截该缺陷。
- 为什么本实验始终不并行 `k` 循环？若必须并行化它，需要付出什么代价？在何种情形下这一代价是值得的（提示：考虑 M、N 很小而 K 极大的情形）。
- `collapse(2)` 将任务数由 16 提高到 256。任务是否越多越好？若将块尺寸减至 8×8（任务数 16384）会产生什么后果？请从**负载均衡**与**算术强度／调度开销**两个方向分别论证。
- 数据竞争以 `64 64 64` 运行时无法复现，以 `1024 1024 1024` 运行时几乎必然出现。这对"如何为并行程序设计测试用例"有何启示？请给出三条具体的测试规则。
- 强扩展性研究为何必须以"同一份代码的单线程运行"为基准？若误用串行标量基准，所得的"并行效率"会是什么数量级？其错误在何处？
- 同一份代码在 $1024^3$ 与 $2048^3$ 下的并行效率并不相同。这说明并行效率是实现的固有属性，还是"实现 + 问题规模 + 线程数"共同决定的量？报告并行效率时应当同时给出哪些信息？

## 16. 小结与后续

本实验在第三章单核优化成果的基础上完成了向多核的扩展：

<!--
| 版本 | 新增内容 | 涉及知识点 | Check |
|---|---|---|---|
| **v1** | `gemm_serial_no_vec` + `gemm_neon_1t` | 两条基准；总加速比与线程级贡献的区分 | PASS |
| **v2** | `gemm_omp_naive` | `parallel for`、`schedule`、**加速比的局限性** | PASS |
| **v3** | `gemm_omp_neon_tiled` | 三条判据、`collapse(2)`、输出可划分带来的便利 | PASS |
| **v4** | `gemm_omp_pack_race` | **数据竞争**：无崩溃、无警告、小规模不复现 | **FAIL** |
| **v5** | `gemm_omp_pack` | 线程私有工作区、`parallel { for }` 的拆分写法 | PASS |
| **🚀 扩展** | `gemm_omp_pack_8x8` | 双打包与多线程结合；分块尺寸与任务数的权衡 | 学生完成 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
      <th style="text-align: left;">Check</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>gemm_serial_no_vec</code> + <code>gemm_neon_1t</code></td>
      <td style="text-align: left;">两条基准；总加速比与线程级贡献的区分</td>
      <td style="text-align: left;">PASS</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>gemm_omp_naive</code></td>
      <td style="text-align: left;"><code>parallel for</code>、<code>schedule</code>、<strong>加速比的局限性</strong></td>
      <td style="text-align: left;">PASS</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>gemm_omp_neon_tiled</code></td>
      <td style="text-align: left;">三条判据、<code>collapse(2)</code>、输出可划分带来的便利</td>
      <td style="text-align: left;">PASS</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v4</strong></td>
      <td style="text-align: left;"><code>gemm_omp_pack_race</code></td>
      <td style="text-align: left;"><strong>数据竞争</strong>：无崩溃、无警告、小规模不复现</td>
      <td style="text-align: left;"><strong>FAIL</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v5</strong></td>
      <td style="text-align: left;"><code>gemm_omp_pack</code></td>
      <td style="text-align: left;">线程私有工作区、<code>parallel { for }</code> 的拆分写法</td>
      <td style="text-align: left;">PASS</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>🚀 扩展</strong></td>
      <td style="text-align: left;"><code>gemm_omp_pack_8x8</code></td>
      <td style="text-align: left;">双打包与多线程结合；分块尺寸与任务数的权衡</td>
      <td style="text-align: left;">学生完成</td>
    </tr>
  </tbody>
</table>

三条主要结论：

1. **优化的顺序应当是先单核、后多核。** 三个并行层级呈相乘关系，单核的每一分收益都会被线程数放大；反之，单核的每一分浪费同样会被放大 $T$ 倍。此外，访存受限的低效实现不仅绝对性能低，其**扩展性同样受限**，因而付出的是双重代价。
2. **并行引入的首要风险不是性能不足，而是结果错误。** 串行代码中"函数内部申请临时缓冲区"是常见写法，进入并行域后即成为共享状态。审查并行代码时，应当**先检查每个函数的内部状态，再检查其并行指令**。
3. **并行程序的缺陷具有概率性。** "测试通过"不等价于"实现正确"——必须以足够大的问题规模、足够多的线程数反复验证，必要时借助 ThreadSanitizer 等工具检测。

> **🎓 概括而言**：SIMD 决定**单个核心能达到的性能上限**，OpenMP 决定**能够使用多少个核心**，而对数据竞争的控制决定**计算结果是否有效**。